# Build a File-Organizer Bot — runnable notebook

This notebook is a Colab/Kaggle/Binder-friendly demo of the **Build a File-Organizer Bot** project from the
[Python & Data Analysis course](https://github.com/abderrahim-lectures/python-data-analysis-course). It mirrors the
real, runnable example at [`examples/file-organizer/organizer.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/file-organizer/organizer.py).

**Important honesty note:** a hosted notebook session (Colab, Kaggle, or Binder) runs on its own temporary machine with its own
filesystem — it has no access to your real computer's Downloads folder, or any other folder on your own machine. So this notebook
cannot organize your real files. Instead, it creates a small **sample messy folder** inside the notebook's own ephemeral filesystem,
runs the organizer on that sample folder (dry-run, then for real, then undo), and everything disappears when the session ends.
That's a genuine demo of how the tool behaves, not a way to organize anything that matters to you — for that, run this project
locally with `uv` (see the [project doc](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/docs/projects/file-organizer/index.md)),
where the script can see your real files.

## Step 1: Define the extension → category mapping

No installs needed — the whole project is Python's standard library (`pathlib`, `shutil`, `json`). Run the cells in order.

In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path

EXTENSION_CATEGORIES: dict[str, str] = {
    "pdf": "documents", "doc": "documents", "docx": "documents", "txt": "documents",
    "md": "documents", "csv": "documents", "xlsx": "documents",
    "jpg": "images", "jpeg": "images", "png": "images", "gif": "images", "svg": "images",
    "zip": "archives", "tar": "archives", "gz": "archives",
    "py": "code", "js": "code", "ts": "code", "html": "code", "css": "code", "json": "code",
    "mp3": "audio", "wav": "audio",
    "mp4": "video", "mov": "video",
}
DEFAULT_CATEGORY = "other"
LOG_FILENAME = ".organizer_log.json"

def category_for(path: Path) -> str:
    ext = path.suffix.lower().lstrip(".")
    return EXTENSION_CATEGORIES.get(ext, DEFAULT_CATEGORY)

## Step 2: Create a sample messy folder (inside this notebook's own filesystem)

This is a **demo folder**, not your real files. It lives at `sample_messy_folder/` in this notebook session's temporary storage
and disappears when the session ends. It deliberately includes a pre-existing `documents/report.pdf`, so organizing a *new*
`report.pdf` triggers a real collision — not a hypothetical one.

In [ ]:
SAMPLE_FILES = [
    ("report.pdf", "Quarterly report placeholder.\n"),
    ("notes.txt", "Random notes.\n"),
    ("photo1.jpg", "not a real jpg, just sample bytes\n"),
    ("archive.zip", "not a real zip, just sample bytes\n"),
    ("script.py", "print('hello from a sample script')\n"),
    ("song.mp3", "not a real mp3, just sample bytes\n"),
    ("mystery", "A file with no extension.\n"),
    (".hidden_config", "A dotfile the organizer should skip.\n"),
]

sample_folder = Path("sample_messy_folder").resolve()
sample_folder.mkdir(exist_ok=True)
for name, contents in SAMPLE_FILES:
    (sample_folder / name).write_text(contents)

documents_dir = sample_folder / "documents"
documents_dir.mkdir(exist_ok=True)
(documents_dir / "report.pdf").write_text("An older report.pdf already living here.\n")

print(f"Sample folder ready at: {sample_folder}")
print(sorted(p.name for p in sample_folder.iterdir()))

## Step 3: Dry run — see the plan without moving anything

In [ ]:
def scan_folder(folder: Path) -> list[Path]:
    return [
        p for p in sorted(folder.iterdir())
        if p.is_file() and not p.name.startswith(".") and p.name != LOG_FILENAME
    ]

def resolve_collision(destination: Path, reserved: set[Path]) -> Path:
    if not destination.exists() and destination not in reserved:
        return destination
    stem, suffix, parent = destination.stem, destination.suffix, destination.parent
    counter = 1
    while True:
        candidate = parent / f"{stem} ({counter}){suffix}"
        if not candidate.exists() and candidate not in reserved:
            return candidate
        counter += 1

def plan_moves(folder: Path) -> list[tuple[Path, Path]]:
    moves = []
    reserved: set[Path] = set()
    for source in scan_folder(folder):
        dest_dir = folder / category_for(source)
        destination = resolve_collision(dest_dir / source.name, reserved)
        reserved.add(destination)
        moves.append((source, destination))
    return moves

moves = plan_moves(sample_folder)
print(f"[DRY RUN] Planned moves ({len(moves)}):")
for source, destination in moves:
    print(f"  {source.relative_to(sample_folder)}  ->  {destination.relative_to(sample_folder)}")

Notice `report.pdf` is planned to land at `documents/report (1).pdf`, not `documents/report.pdf` — that's the collision handling
kicking in against the pre-existing file from Step 2.

## Step 4: Actually move the files, and write an operations log

In [ ]:
def apply_moves(moves: list[tuple[Path, Path]], folder: Path) -> list[dict]:
    log_entries = []
    for source, destination in moves:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(source), str(destination))
        log_entries.append({
            "source": str(source.relative_to(folder)),
            "destination": str(destination.relative_to(folder)),
        })
        print(f"Moved: {source.relative_to(folder)}  ->  {destination.relative_to(folder)}")
    return log_entries

log_entries = apply_moves(moves, sample_folder)
log_path = sample_folder / LOG_FILENAME
log_path.write_text(json.dumps({
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "moves": log_entries,
}, indent=2))
print(f"\nOperations log written to {log_path.name}")

## Step 5: Undo — reverse the run using the operations log

In [ ]:
def undo_last_run(folder: Path) -> None:
    log_path = folder / LOG_FILENAME
    run = json.loads(log_path.read_text())
    for entry in reversed(run["moves"]):
        destination = folder / entry["destination"]
        source = folder / entry["source"]
        source.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(destination), str(source))
        print(f"Restored: {entry['destination']}  ->  {entry['source']}")
    log_path.unlink()

undo_last_run(sample_folder)
print("\nUndo complete.")
print(sorted(p.name for p in sample_folder.iterdir()))

## Where to go from here

This notebook mirrors `organizer.py`'s core logic in a demo-friendly way, but the real project (with a full CLI, `--dry-run`
and `--undo` flags, and better collision/error handling) lives at
[`examples/file-organizer/organizer.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/file-organizer/organizer.py).
Run it locally with `uv` against a real folder (after generating and testing on a sample one first!) — see the
[full lesson](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/docs/projects/file-organizer/index.md).